In [3]:
# ── CELL 1: AUTHENTICATION AND GCS BUCKET SETUP ────────
# Runtime: CPU (default Colab runtime)
# This cell takes about 2 minutes

from google.colab import drive, auth
import subprocess
import os

# Mount your Google Drive
drive.mount('/content/drive')

# Authenticate for Google Cloud (GCS, GEE)
auth.authenticate_user()

# ── CONFIG — edit these to match your setup ────────────
GCP_PROJECT   = 'integrated-hawk-485001-k3'  # Your GCP project
GCS_BUCKET    = 'gs://tala-sentinel2-data'   # Name must be globally unique
GCS_REGION    = 'us-central1'               # Closest to your Drive data

# ── CREATE BUCKET ──────────────────────────────────────
print("Creating GCS bucket...")
result = subprocess.run([
    'gsutil', 'mb',
    '-p', GCP_PROJECT,
    '-l', GCS_REGION,
    '-b', 'on',          # Uniform bucket-level access
    GCS_BUCKET
], capture_output=True, text=True)

# "already exists" is not an error
if 'already exists' in result.stderr:
    print(f"Bucket already exists: {GCS_BUCKET}")
elif result.returncode == 0:
    print(f"Bucket created: {GCS_BUCKET}")
else:
    print(f"Error: {result.stderr}")

# ── CREATE SUBDIRECTORIES ──────────────────────────────
subdirs = [
    'tfrecords/sentinel2',
    'tfrecords/viirs',
    'models/proxy_cnn',
    'models/lstm',
    'features/dynamic',
    'features/static',
    'logs/proxy_cnn',
    'logs/lstm',
    'exports/output_provinces'
]

for subdir in subdirs:
    subprocess.run([
        'gsutil', 'cp', '/dev/null',
        f'{GCS_BUCKET}/{subdir}/.keep'
    ], capture_output=True)

print("\nGCS structure created:")
result = subprocess.run(
    ['gsutil', 'ls', GCS_BUCKET],
    capture_output=True, text=True
)
print(result.stdout)
print("✓ Cell 1 complete")

Mounted at /content/drive
Creating GCS bucket...
Bucket already exists: gs://tala-sentinel2-data

GCS structure created:
gs://tala-sentinel2-data/exports/
gs://tala-sentinel2-data/features/
gs://tala-sentinel2-data/logs/
gs://tala-sentinel2-data/models/
gs://tala-sentinel2-data/tfrecords/

✓ Cell 1 complete


In [ ]:
# ── CELL 2: VERIFY DRIVE CONTENTS ─────────────────────
# This confirms your files are where the conversion expects them
# Takes about 1 minute

import os
import pandas as pd

# ── EDIT THESE PATHS TO MATCH YOUR DRIVE ──────────────
SENTINEL_DIR  = '/content/drive/MyDrive/Sentinel2_Training_Data'
LABELS_CSV    = '/content/drive/MyDrive/Thesis_Data/viirs_ntl_labels_all_clusters.csv'

# Check Sentinel directory
if os.path.exists(SENTINEL_DIR):
    all_tifs = [
        f for f in os.listdir(SENTINEL_DIR)
        if f.endswith('.tif') and 'dhs_' in f
    ]
    print(f"✓ Sentinel directory found")
    print(f"  Total .tif files: {len(all_tifs)}")

    # Sample a few filenames to confirm naming format
    print(f"  Sample filenames:")
    for f in sorted(all_tifs)[:5]:
        print(f"    {f}")
else:
    print(f"✗ Sentinel directory NOT found: {SENTINEL_DIR}")
    print("  Update SENTINEL_DIR to match your Drive path")

print()

# Check labels CSV
if os.path.exists(LABELS_CSV):
    df_labels = pd.read_csv(LABELS_CSV)
    print(f"✓ Labels CSV found")
    print(f"  Rows: {len(df_labels)}")
    print(f"  Columns: {list(df_labels.columns)}")
    print(f"  Class distribution:")
    print(df_labels['NTL_Class'].value_counts().sort_index().to_string())
else:
    print(f"✗ Labels CSV NOT found: {LABELS_CSV}")
    print("  Run phase1-ntl-median.ipynb first to generate this file")

print()

# Cross-check: how many TIF clusters have a label
if os.path.exists(SENTINEL_DIR) and os.path.exists(LABELS_CSV):
    label_ids = set(df_labels['DHSCLUST'].astype(int))
    tif_ids   = set()
    for f in all_tifs:
        try:
            parts = f.replace('.tif','').split('_')
            tif_ids.add(int(parts[1]))
        except:
            pass
    matched = tif_ids & label_ids
    print(f"Cluster IDs in TIFs:   {len(tif_ids)}")
    print(f"Cluster IDs in labels: {len(label_ids)}")
    print(f"Matched (will convert):{len(matched)}")
    unmatched = tif_ids - label_ids
    if unmatched:
        print(f"TIF clusters with no label (will skip): {len(unmatched)}")
        print(f"  Sample: {sorted(list(unmatched))[:10]}")

✓ Sentinel directory found
  Total .tif files: 4988
  Sample filenames:
    dhs_1000_2022_Q1.tif
    dhs_1000_2022_Q2.tif
    dhs_1000_2022_Q3.tif
    dhs_1000_2022_Q4.tif
    dhs_1001_2022_Q1.tif

✓ Labels CSV found
  Rows: 1247
  Columns: ['DHSCLUST', 'URBAN_RURA', 'NTL_Value', 'NTL_Class']
  Class distribution:
NTL_Class
0    416
1    415
2    416

Cluster IDs in TIFs:   1247
Cluster IDs in labels: 1247
Matched (will convert):1247


In [ ]:
# ── CELL 3: TFRECORD CONVERSION ────────────────────────
# This is the long-running cell. Start it and leave it.
# It saves shards directly to GCS as it goes.
# If it disconnects, re-run — it skips already-written shards.
# Takes 4 to 6 hours for ~5000 files

import os
import re
import time
import numpy as np
import pandas as pd
import rasterio
import tensorflow as tf
from datetime import datetime

# ── PATHS ──────────────────────────────────────────────
SENTINEL_DIR  = '/content/drive/MyDrive/Sentinel2_Training_Data'
#LABELS_CSV    = '/content/drive/MyDrive/viirs_ntl_labels_all_clusters.csv'
GCS_BUCKET    = 'gs://tala-sentinel2-data'
GCS_TFRECORDS = f'{GCS_BUCKET}/tfrecords/sentinel2'

TARGET_SIZE   = (224, 224)
N_SHARDS      = 32        # 32 shards, ~156 images per shard
LOG_EVERY     = 50        # Print progress every N images

# ── LOAD LABELS ────────────────────────────────────────
df_labels  = pd.read_csv(LABELS_CSV)
label_map  = dict(zip(
    df_labels['DHSCLUST'].astype(int),
    df_labels['NTL_Class'].astype(int)
))
print(f"Loaded {len(label_map)} cluster labels")

# ── COLLECT ALL VALID PATHS ────────────────────────────
print("Scanning Sentinel directory...")
all_paths, all_labels, all_cluster_ids, all_quarters = [], [], [], []

for fname in sorted(os.listdir(SENTINEL_DIR)):
    if not fname.endswith('.tif') or 'dhs_' not in fname:
        continue
    try:
        parts      = fname.replace('.tif', '').split('_')
        cluster_id = int(parts[1])
        quarter    = parts[3]   # Q1, Q2, Q3, Q4
        if cluster_id not in label_map:
            continue
        all_paths.append(os.path.join(SENTINEL_DIR, fname))
        all_labels.append(label_map[cluster_id])
        all_cluster_ids.append(cluster_id)
        all_quarters.append(int(quarter.replace('Q', '')))
    except Exception:
        continue

total_images = len(all_paths)
print(f"Found {total_images} valid images to convert")

# ── IMAGE LOADING ──────────────────────────────────────
def load_and_normalize(path):
    """
    Loads a Sentinel-2 GeoTIFF, extracts RGB bands,
    applies percentile stretch normalization, resizes to 224x224.
    Returns float32 numpy array or None on error.
    """
    try:
        with rasterio.open(path) as src:
            # Read first 3 bands (B4=Red, B3=Green, B2=Blue)
            # Your GeoTIFFs have bands: B4, B3, B2, B8, B11
            r = src.read(1).astype(np.float32)
            g = src.read(2).astype(np.float32)
            b = src.read(3).astype(np.float32)
            img = np.dstack((r, g, b))

            # Percentile stretch (same as phase1-proxy-cnn.ipynb)
            p2, p98 = np.percentile(img, (2, 98))
            if p98 <= p2:
                p98 = p2 + 1e-6
            img = np.clip((img - p2) / (p98 - p2), 0, 1)

            # Resize to 224x224 for VGG16
            img_tensor = tf.image.resize(img, TARGET_SIZE)
            return img_tensor.numpy().astype(np.float32)
    except Exception as e:
        return None

# ── TFRECORD HELPERS ───────────────────────────────────
def bytes_feature(val):
    return tf.train.Feature(
        bytes_list=tf.train.BytesList(value=[val]))

def int64_feature(val):
    return tf.train.Feature(
        int64_list=tf.train.Int64List(value=[val]))

def make_example(img, label, cluster_id, quarter):
    return tf.train.Example(features=tf.train.Features(feature={
        'image'      : bytes_feature(img.tobytes()),
        'label'      : int64_feature(label),
        'cluster_id' : int64_feature(cluster_id),
        'quarter'    : int64_feature(quarter),
        # Store height/width so we can reshape without hardcoding
        'height'     : int64_feature(TARGET_SIZE[0]),
        'width'      : int64_feature(TARGET_SIZE[1]),
        'channels'   : int64_feature(3),
    }))

# ── CHECK WHICH SHARDS ALREADY EXIST ──────────────────
print("\nChecking for existing shards on GCS...")
existing = set()
try:
    result = subprocess.run(
        ['gsutil', 'ls', f'{GCS_TFRECORDS}/'],
        capture_output=True, text=True
    )
    for line in result.stdout.strip().split('\n'):
        if '.tfrecord' in line:
            fname = line.strip().split('/')[-1]
            # Extract shard index from filename
            match = re.search(r'shard_(\d+)', fname)
            if match:
                existing.add(int(match.group(1)))
    print(f"Found {len(existing)} existing shards, will skip them")
except Exception:
    print("No existing shards found, starting fresh")

# ── CONVERSION LOOP ────────────────────────────────────
shard_size   = (total_images // N_SHARDS) + 1
written      = 0
skipped_imgs = 0
start_time   = time.time()

print(f"\nStarting conversion: {total_images} images → {N_SHARDS} shards")
print(f"Shard size: ~{shard_size} images each")
print(f"Output: {GCS_TFRECORDS}")
print("="*60)

for shard_idx in range(N_SHARDS):

    # Skip if shard already written
    if shard_idx in existing:
        shard_start = shard_idx * shard_size
        shard_end   = min(shard_start + shard_size, total_images)
        print(f"Shard {shard_idx:03d}: SKIPPED (already on GCS, "
              f"{shard_end - shard_start} images)")
        written += (shard_end - shard_start)
        continue

    shard_path  = f'{GCS_TFRECORDS}/shard_{shard_idx:03d}.tfrecord'
    shard_start = shard_idx * shard_size
    shard_end   = min(shard_start + shard_size, total_images)
    shard_count = 0
    shard_skip  = 0

    with tf.io.TFRecordWriter(shard_path) as writer:
        for i in range(shard_start, shard_end):
            img = load_and_normalize(all_paths[i])

            if img is None:
                skipped_imgs += 1
                shard_skip   += 1
                continue

            ex = make_example(
                img,
                all_labels[i],
                all_cluster_ids[i],
                all_quarters[i]
            )
            writer.write(ex.SerializeToString())
            shard_count += 1
            written     += 1

            if written % LOG_EVERY == 0:
                elapsed = time.time() - start_time
                rate    = written / elapsed
                eta_sec = (total_images - written) / rate if rate > 0 else 0
                eta_min = eta_sec / 60
                print(f"  [{datetime.now().strftime('%H:%M:%S')}] "
                      f"{written}/{total_images} images "
                      f"({written/total_images*100:.1f}%) | "
                      f"{rate:.1f} img/s | "
                      f"ETA: {eta_min:.0f} min")

    print(f"Shard {shard_idx:03d}: ✓ wrote {shard_count} "
          f"(skipped {shard_skip})")

# ── FINAL SUMMARY ──────────────────────────────────────
total_elapsed = (time.time() - start_time) / 60
print("\n" + "="*60)
print("CONVERSION COMPLETE")
print("="*60)
print(f"  Images written : {written}")
print(f"  Images skipped : {skipped_imgs} (corrupt or unreadable)")
print(f"  Total time     : {total_elapsed:.1f} minutes")
print(f"  Output location: {GCS_TFRECORDS}")

# Verify final file count on GCS
result = subprocess.run(
    ['gsutil', 'du', '-sh', f'{GCS_TFRECORDS}/'],
    capture_output=True, text=True
)
print(f"  Total GCS size : {result.stdout.strip()}")

Loaded 1247 cluster labels
Scanning Sentinel directory...
Found 4988 valid images to convert

Checking for existing shards on GCS...
Found 0 existing shards, will skip them

Starting conversion: 4988 images → 32 shards
Shard size: ~156 images each
Output: gs://tala-sentinel2-data/tfrecords/sentinel2
Shard 000: ✓ wrote 49 (skipped 107)
Shard 001: ✓ wrote 0 (skipped 156)
  [07:27:08] 50/4988 images (1.0%) | 0.1 img/s | ETA: 719 min
  [07:27:36] 100/4988 images (2.0%) | 0.2 img/s | ETA: 378 min
  [07:27:58] 150/4988 images (3.0%) | 0.3 img/s | ETA: 262 min
  [07:28:37] 200/4988 images (4.0%) | 0.4 img/s | ETA: 210 min
Shard 002: ✓ wrote 156 (skipped 0)
  [07:29:05] 250/4988 images (5.0%) | 0.5 img/s | ETA: 175 min
  [07:29:44] 300/4988 images (6.0%) | 0.5 img/s | ETA: 154 min
  [07:30:11] 350/4988 images (7.0%) | 0.6 img/s | ETA: 137 min
Shard 003: ✓ wrote 156 (skipped 0)
  [07:30:36] 400/4988 images (8.0%) | 0.6 img/s | ETA: 123 min
  [07:31:07] 450/4988 images (9.0%) | 0.7 img/s | ETA: 

In [16]:
# ── CELL 4: VERIFY TFRECORDS ───────────────────────────
# Run this after conversion finishes to confirm
# the TFRecords are valid before switching to TPU runtime

import tensorflow as tf

GCS_BUCKET    = 'gs://tala-sentinel2-data'
GCS_TFRECORDS = f'{GCS_BUCKET}/tfrecords/sentinel2'

feature_spec = {
    'image'     : tf.io.FixedLenFeature([], tf.string),
    'label'     : tf.io.FixedLenFeature([], tf.int64),
    'cluster_id': tf.io.FixedLenFeature([], tf.int64),
    'quarter'   : tf.io.FixedLenFeature([], tf.int64),
    'height'    : tf.io.FixedLenFeature([], tf.int64),
    'width'     : tf.io.FixedLenFeature([], tf.int64),
    'channels'  : tf.io.FixedLenFeature([], tf.int64),
}

def parse(example_proto):
    parsed = tf.io.parse_single_example(example_proto, feature_spec)
    h      = tf.cast(parsed['height'],   tf.int32)
    w      = tf.cast(parsed['width'],    tf.int32)
    c      = tf.cast(parsed['channels'], tf.int32)
    img    = tf.io.decode_raw(parsed['image'], tf.float32)
    img    = tf.reshape(img, [h, w, c])
    return img, parsed['label'], parsed['cluster_id'], parsed['quarter']

# List all shards
shards = tf.io.gfile.glob(f'{GCS_TFRECORDS}/*.tfrecord')
print(f"Shards found on GCS: {len(shards)}")

# Read first 3 records from first shard
print("\nReading sample records...")
ds = tf.data.TFRecordDataset(shards[:1])
for i, raw in enumerate(ds.take(3)):
    img, label, cid, quarter = parse(raw)
    print(f"\nRecord {i+1}:")
    print(f"  Image shape : {img.shape}")
    print(f"  Dtype       : {img.dtype}")
    print(f"  Pixel range : [{img.numpy().min():.3f}, "
          f"{img.numpy().max():.3f}]")
    print(f"  Label       : {label.numpy()} "
          f"(0=Dark, 1=Dim, 2=Bright)")
    print(f"  Cluster ID  : {cid.numpy()}")
    print(f"  Quarter     : Q{quarter.numpy()}")

# Count total records across all shards
print("\nCounting total records across all shards...")
print("(This takes a few minutes)")
total = 0
for shard in shards:
    ds_count = tf.data.TFRecordDataset(shard)
    total   += sum(1 for _ in ds_count)
print(f"\nTotal records in TFRecords: {total}")
print(f"Expected: ~{len(shards) * 156} (based on shard count)")

# Class distribution check
print("\nChecking class distribution across first 5 shards...")
from collections import Counter
label_counts = Counter()
ds_sample = tf.data.TFRecordDataset(shards[:5])
for raw in ds_sample:
    img, label, cid, q = parse(raw)
    label_counts[int(label.numpy())] += 1

total_sample = sum(label_counts.values())
print(f"  Dark  (0): {label_counts[0]} "
      f"({label_counts[0]/total_sample*100:.1f}%)")
print(f"  Dim   (1): {label_counts[1]} "
      f"({label_counts[1]/total_sample*100:.1f}%)")
print(f"  Bright(2): {label_counts[2]} "
      f"({label_counts[2]/total_sample*100:.1f}%)")
print("\n✓ TFRecords verified. Ready for TPU training.")
print(f"\nNext step: open your TPU Colab notebook and point it to:")
print(f"  {GCS_TFRECORDS}")

Shards found on GCS: 33

Reading sample records...

Record 1:
  Image shape : (224, 224, 3)
  Dtype       : <dtype: 'float32'>
  Pixel range : [0.000, 1.000]
  Label       : 1 (0=Dark, 1=Dim, 2=Bright)
  Cluster ID  : 1000
  Quarter     : Q1

Record 2:
  Image shape : (224, 224, 3)
  Dtype       : <dtype: 'float32'>
  Pixel range : [0.000, 1.000]
  Label       : 1 (0=Dark, 1=Dim, 2=Bright)
  Cluster ID  : 1000
  Quarter     : Q2

Record 3:
  Image shape : (224, 224, 3)
  Dtype       : <dtype: 'float32'>
  Pixel range : [0.000, 1.000]
  Label       : 1 (0=Dark, 1=Dim, 2=Bright)
  Cluster ID  : 1000
  Quarter     : Q3

Counting total records across all shards...
(This takes a few minutes)

Total records in TFRecords: 4939
Expected: ~5148 (based on shard count)

Checking class distribution across first 5 shards...
  Dark  (0): 168 (32.5%)
  Dim   (1): 204 (39.5%)
  Bright(2): 145 (28.0%)

✓ TFRecords verified. Ready for TPU training.

Next step: open your TPU Colab notebook and point it t

In [17]:
# ── CONFIRM ALL SHARDS ON GCS BEFORE SWITCHING RUNTIME ─
import subprocess
import tensorflow as tf

GCS_TFRECORDS = 'gs://tala-sentinel2-data/tfrecords/sentinel2'

# List all shards
result = subprocess.run(
    ['gsutil', 'ls', '-l', f'{GCS_TFRECORDS}/'],
    capture_output=True, text=True
)
lines  = [l for l in result.stdout.strip().split('\n')
          if '.tfrecord' in l]
n_shards = len(lines)

print(f"Shards on GCS : {n_shards}")
print(f"Expected      : 34 (shards 000-033)")

# Total size
result = subprocess.run(
    ['gsutil', 'du', '-sh', f'{GCS_TFRECORDS}/'],
    capture_output=True, text=True
)
print(f"Total size    : {result.stdout.strip()}")

# Quick record count
shards = tf.io.gfile.glob(f'{GCS_TFRECORDS}/*.tfrecord')
total  = sum(1 for s in shards
             for _ in tf.data.TFRecordDataset(s))
print(f"Total records : {total}")
print(f"Expected      : 4988")

if total == 4988:
    print("\n✓ All records confirmed. Safe to switch to TPU runtime.")
else:
    print(f"\n⚠ Gap of {4988 - total}. Check before proceeding.")

Shards on GCS : 33
Expected      : 34 (shards 000-033)
Total size    : 2.77 GiB     gs://tala-sentinel2-data/tfrecords/sentinel2
Total records : 4939
Expected      : 4988

⚠ Gap of 49. Check before proceeding.
